In [18]:
# Librerias

import hashlib, json
from bitcoinutils.setup import setup
from bitcoinutils.keys import PrivateKey, PublicKey, P2trAddress
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.script import Script
from coincurve import PrivateKey as ccPrivateKey , PublicKeyXOnly 
from pathlib import Path
from bitcoinutils.utils import tapleaf_tagged_hash, tapbranch_tagged_hash

setup("regtest")

'regtest'

In [33]:
# Preparando el Batch de PDFs

def hash_pdf(pdf_path):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    
    return hashlib.sha256(pdf_bytes).hexdigest()

def extract_id_from_pdf_name(pdf_path):
    name = Path(pdf_path).stem   # por ejemplo: doc_001_3d198
    return name.replace("doc_", "", 1)

def sort_hash_dict(batch):
    return dict(
        sorted(
            ((int(k.split("_")[0]), v) for k, v in batch.items()),
            key=lambda item: item[0]
        )
    )

def hash_pdfs_in_path(folder_path: str):
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"La ruta no existe: {folder_path}")
    if not folder.is_dir():
        raise NotADirectoryError(f"La ruta no es un directorio: {folder_path}")
    hashes = {}
    for pdf_file in folder.glob("*.pdf"):
        index = extract_id_from_pdf_name(pdf_file)
        hashes[index[:-6]] = hash_pdf(pdf_file)
    return hashes

N_batch = 8
batch = sort_hash_dict(hash_pdfs_in_path("files/batch_size/"+str(N_batch)))
batch

{1: 'ba109d0938281a88ec8fd82c162f0523852fd2201ca2d7d48eb50b6fa73e8a9e',
 2: '018e3751a083e2c7551381d9ef1d94276f19d8baf53ca9b8944b69d844a5e908',
 3: 'b868af59633828ff160beb0332adc85e4938881f15bf0decf88b2d7666ed5831',
 4: '37d0680346c3e9a2bdbe1ffda7a1e73cf9078642b3ad2e78e7fc46b585e13537',
 5: 'c88e67aab82fa6242c02387df0b34118b0245f397e47499daf12cc6d8fe259a2',
 6: '4e0c9f9ed2730eaa60b4363827187cabd1834d4855cb622d07b85c6daea2c75f',
 7: '442bc8c8ee3d9fd9708f60f13e4be0e654c37aafca4ca5c358a990880d65a4a0',
 8: '1c33891e8e92e3c09cffa29718e5d1f968e0d2cb8c202ce48b99e0161f698c58'}

In [34]:
# Preparando el setup Taproot

seed_sign_key = hashlib.sha256(b"Deanship").digest()
sk_sign = ccPrivateKey(seed_sign_key)
pk_sign = sk_sign.public_key_xonly.format().hex()

CONTROL = {}
for key, value in batch.items():
    doc_sing = sk_sign.sign_schnorr(bytes.fromhex(value))
    CONTROL[key] = {
                    "index": key,
                    "sig": doc_sing.hex(),
                    "pk_sign": pk_sign,
                    "hash": value,
                    }
CONTROL[1]

{'index': 1,
 'sig': '79687e69764c27d136a2c369ddbc974a3ff96640c396c5bf9fe3ad61393648a7b947af842dca29a2a1344f83de3fcf4df68e7964a1bbac224bb5e57abc2a5149',
 'pk_sign': '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3',
 'hash': 'ba109d0938281a88ec8fd82c162f0523852fd2201ca2d7d48eb50b6fa73e8a9e'}

In [35]:
# Verificación
A = PublicKeyXOnly(bytes.fromhex(CONTROL[1]['pk_sign']))
B = bytes.fromhex(CONTROL[1]['sig'])
C = bytes.fromhex(CONTROL[1]['hash'])
A.verify(B,C)

True

In [36]:
# Preparando el setup Taproot 

btc_sk = PrivateKey(secret_exponent=1)
btc_pub = btc_sk.get_public_key()
btc_xonly = btc_pub.to_x_only_hex()

leafs = []

for value in CONTROL.values():
    index = int(value['index']).to_bytes(4, "big").hex()
    hash_doc = value['hash']
    sign = value['pk_sign']
    opcode = 'OP_FALSE'
    script = Script([index,hash_doc,sign,opcode])
    leafs.append(script)
    
leafs[0]

['00000001', 'ba109d0938281a88ec8fd82c162f0523852fd2201ca2d7d48eb50b6fa73e8a9e', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE']

In [58]:
def build_tap_tree(leafs):
    nodes = leafs[:]   

    while len(nodes) > 1:
        next_level = []

        for i in range(0, len(nodes), 2):
            if i + 1 < len(nodes):
                next_level.append([nodes[i], nodes[i + 1]])
            else:
                next_level.append(nodes[i])

        nodes = next_level

    return nodes[0]

tap_tree = build_tap_tree(leafs)
taproot_addr = btc_pub.get_taproot_address(tap_tree)
COMMIT_ADDRESS = taproot_addr
print(taproot_addr.to_string())
print(type(tap_tree[0][0]))


bcrt1p0ed8pgcm2wh6k26je56u4jk3p38wtm7vhz2tr607k8ux6enmp9kqv2u5cl
<class 'list'>


In [38]:
# construyendo el Proor of verificatión.

def build_merkle_proofs_from_leafs(leafs):
    if not leafs:
        raise ValueError("leafs está vacío")

    # hash de cada hoja, en el mismo orden que leafs
    level = [
        {
            "hash": tapleaf_tagged_hash(leaf),
            "positions": [i]
        }
        for i, leaf in enumerate(leafs)
    ]

    proofs = {i: [] for i in range(len(leafs))}

    while len(level) > 1:
        next_level = []

        for i in range(0, len(level), 2):
            if i + 1 < len(level):
                left = level[i]
                right = level[i + 1]

                parent_hash = tapbranch_tagged_hash(left["hash"], right["hash"])

                for pos in left["positions"]:
                    proofs[pos].append(right["hash"])

                for pos in right["positions"]:
                    proofs[pos].append(left["hash"])

                next_level.append({
                    "hash": parent_hash,
                    "positions": left["positions"] + right["positions"]
                })
            else:
                # promoción directa si el nivel tiene cantidad impar
                next_level.append(level[i])

        level = next_level

    root = level[0]["hash"]
    return root, proofs


def build_pdf_deliverables(CONTROL, leafs, internal_key_hex, taproot_addr_str=None):
    if len(CONTROL) != len(leafs):
        raise ValueError("CONTROL y leafs no tienen la misma longitud")

    root, proofs = build_merkle_proofs_from_leafs(leafs)

    deliverables = {}

    # importante: mismo orden que se usó al construir leafs
    control_items = list(CONTROL.items())

    for pos, (doc_id, item) in enumerate(control_items):
        deliverables[doc_id] = {
            "index": item["index"],
            "hash": item["hash"],
            "sig": item["sig"],
            "pk_sign": item["pk_sign"],
            "proof": [h.hex() for h in proofs[pos]],
            "internal_key": internal_key_hex,
            "root": root.hex(),
        }

        if taproot_addr_str is not None:
            deliverables[doc_id]["address"] = taproot_addr_str
        filename = f"doc_{doc_id}_proof.json"
        Path("files/batch_size/"+str(N_batch)+'/'+filename).write_text(json.dumps(deliverables, indent=4))
        

    return deliverables

In [39]:
deliverables = build_pdf_deliverables(
    CONTROL=CONTROL,
    leafs=leafs,
    internal_key_hex=btc_xonly,
    taproot_addr_str=taproot_addr.to_string()
)

deliverables[1]

{'index': 1,
 'hash': 'ba109d0938281a88ec8fd82c162f0523852fd2201ca2d7d48eb50b6fa73e8a9e',
 'sig': '79687e69764c27d136a2c369ddbc974a3ff96640c396c5bf9fe3ad61393648a7b947af842dca29a2a1344f83de3fcf4df68e7964a1bbac224bb5e57abc2a5149',
 'pk_sign': '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3',
 'proof': ['93f2dcf4477642516d34e77d5bc7fae0980d9201074d0c207ac14edb7be33709',
  '2368c7ee0258aa450583997840566840024782a89df4a1981f246c05b91ae180',
  '002133c7ae06470949c02c9131b060e6480418a07000dc4474324ba3cd6416b1'],
 'internal_key': '79be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798',
 'root': '79e5a20b482482adfb85625494beaa5dbfd19e475b2e774bb45cbbc5f9daa681',
 'address': 'bcrt1p0ed8pgcm2wh6k26je56u4jk3p38wtm7vhz2tr607k8ux6enmp9kqv2u5cl'}

In [40]:
import json
from pathlib import Path

def deliverable_size_bytes(deliverable) -> int:
    blob = json.dumps(deliverable, separators=(",", ":"), sort_keys=True).encode("utf-8")
    return len(blob)

print(deliverable_size_bytes(deliverables[1]))

743


In [41]:
def rebuild_address_from_deliverable(deliverable):
    required = {
        "index",
        "hash",
        "pk_sign",
        "proof",
        "internal_key",
    }

    missing = required - set(deliverable.keys())
    if missing:
        raise KeyError(f"Faltan campos requeridos: {sorted(missing)}")

    index = deliverable["index"]
    hash_hex = deliverable["hash"]
    pk_sign_hex = deliverable["pk_sign"]
    proof_hex = deliverable["proof"]
    internal_key_hex = deliverable["internal_key"]

    if not isinstance(index, int):
        raise TypeError("deliverable['index'] debe ser int")

    if not isinstance(hash_hex, str) or len(hash_hex) != 64:
        raise ValueError("deliverable['hash'] debe ser hex string de 32 bytes")

    if not isinstance(pk_sign_hex, str) or len(pk_sign_hex) != 64:
        raise ValueError("deliverable['pk_sign'] debe ser x-only hex string de 32 bytes")

    if not isinstance(internal_key_hex, str) or len(internal_key_hex) != 64:
        raise ValueError("deliverable['internal_key'] debe ser x-only hex string de 32 bytes")

    if not isinstance(proof_hex, list):
        raise TypeError("deliverable['proof'] debe ser list[str]")

    for i, sibling_hex in enumerate(proof_hex):
        if not isinstance(sibling_hex, str) or len(sibling_hex) != 64:
            raise ValueError(f"proof[{i}] debe ser hex string de 32 bytes")

    # reconstrucción exacta del leaf
    index_hex = index.to_bytes(4, "big").hex()
    leaf = Script([index_hex, hash_hex, pk_sign_hex, "OP_FALSE"])

    # tapleaf hash
    h = tapleaf_tagged_hash(leaf)
    tapleaf_hash_hex = h.hex()

    # root reconstruida
    for sibling_hex in proof_hex:
        sibling = bytes.fromhex(sibling_hex)
        h = tapbranch_tagged_hash(h, sibling)

    root_rebuilt_hex = h.hex()

    # chequeo opcional contra root guardada
    if "root" in deliverable and deliverable["root"] != root_rebuilt_hex:
        raise ValueError(
            f"Root reconstruida distinta.\n"
            f"esperada={deliverable['root']}\n"
            f"obtenida={root_rebuilt_hex}"
        )

    # internal_key es x-only; para PublicKey.from_hex se reconstruye como compressed even-y
    internal_pub = PublicKey.from_hex("02" + internal_key_hex)

    # python-bitcoin-utils acepta bytes como merkle root en get_taproot_address(...)
    address_obj = internal_pub.get_taproot_address(h)
    address_str = address_obj.to_string()

    # chequeo opcional contra address guardada
    if "address" in deliverable and deliverable["address"] != address_str:
        raise ValueError(
            f"Address reconstruida distinta.\n"
            f"esperada={deliverable['address']}\n"
            f"obtenida={address_str}"
        )

    return {
        "leaf": leaf,
        "tapleaf_hash": tapleaf_hash_hex,
        "root": root_rebuilt_hex,
        "address": address_str,
    }
    
tap_tree = build_tap_tree(leafs)
taproot_addr = btc_pub.get_taproot_address(tap_tree)
print(taproot_addr.to_string())
tap_tree

bcrt1p0ed8pgcm2wh6k26je56u4jk3p38wtm7vhz2tr607k8ux6enmp9kqv2u5cl


[[[['00000001', 'ba109d0938281a88ec8fd82c162f0523852fd2201ca2d7d48eb50b6fa73e8a9e', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE'],
   ['00000002', '018e3751a083e2c7551381d9ef1d94276f19d8baf53ca9b8944b69d844a5e908', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE']],
  [['00000003', 'b868af59633828ff160beb0332adc85e4938881f15bf0decf88b2d7666ed5831', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE'],
   ['00000004', '37d0680346c3e9a2bdbe1ffda7a1e73cf9078642b3ad2e78e7fc46b585e13537', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE']]],
 [[['00000005', 'c88e67aab82fa6242c02387df0b34118b0245f397e47499daf12cc6d8fe259a2', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE'],
   ['00000006', '4e0c9f9ed2730eaa60b4363827187cabd1834d4855cb622d07b85c6daea2c75f', '07fa8ec2c128e00c14a553ddd09067b312c165313dfc96318388f5a70d6542d3', 'OP_FALSE']],
  [['0

In [42]:
result = rebuild_address_from_deliverable(deliverables[1])

print("address reconstructed:", result["address"])
print('Verification: ',COMMIT_ADDRESS.to_string() == result["address"])

address reconstructed: bcrt1p0ed8pgcm2wh6k26je56u4jk3p38wtm7vhz2tr607k8ux6enmp9kqv2u5cl
Verification:  True


In [43]:
# import time
# import statistics

# def benchmark_verification(deliverable, verify_fn, rounds=30, inner_loops=1000, warmup=5):
#     for _ in range(warmup):
#         for _ in range(inner_loops):
#             verify_fn(deliverable)

#     timings = []

#     for _ in range(rounds):
#         t0 = time.perf_counter()
#         for _ in range(inner_loops):
#             verify_fn(deliverable)
#         t1 = time.perf_counter()

#         avg_ms = ((t1 - t0) / inner_loops) * 1000
#         timings.append(avg_ms)

#     return {
#         "mean_ms": round(statistics.mean(timings),2),
#         "stdev_ms": round(statistics.stdev(timings),2) if rounds > 1 else 0.0,
#     }
    
# stats = benchmark_verification(
#     deliverable=deliverables[1],
#     verify_fn=rebuild_address_from_deliverable,
#     rounds=100,
#     inner_loops=10,   
#     warmup=20,
# )

# print(stats)

In [44]:
import time
import statistics
from bitcoinutils.script import Script
from bitcoinutils.utils import tapleaf_tagged_hash, tapbranch_tagged_hash


def rebuild_root_from_deliverable(deliverable):
    index = deliverable["index"]
    hash_hex = deliverable["hash"]
    pk_sign_hex = deliverable["pk_sign"]
    proof_hex = deliverable["proof"]

    index_hex = index.to_bytes(4, "big").hex()

    leaf = Script([
        index_hex,
        hash_hex,
        pk_sign_hex,
        "OP_FALSE",
    ])

    h = tapleaf_tagged_hash(leaf)

    for sibling_hex in proof_hex:
        sibling = bytes.fromhex(sibling_hex)
        h = tapbranch_tagged_hash(h, sibling)

    return h.hex()

def benchmark_proof_only(deliverable, rounds=100, inner_loops=1000, warmup=30):
    for _ in range(warmup):
        for _ in range(inner_loops):
            rebuild_root_from_deliverable(deliverable)

    timings = []

    for _ in range(rounds):
        t0 = time.perf_counter()

        for _ in range(inner_loops):
            rebuild_root_from_deliverable(deliverable)

        t1 = time.perf_counter()

        avg_ms = ((t1 - t0) / inner_loops) * 1000
        timings.append(avg_ms)

    return {
        "mean_ms": round(statistics.mean(timings),4),
        "stdev_ms": round(statistics.stdev(timings),4) if rounds > 1 else 0.0,
    }
    
stats = benchmark_proof_only(
    deliverable=deliverables[1],
    rounds=500,
    inner_loops=1000,
    warmup=100,
)

print(stats)

{'mean_ms': 0.0133, 'stdev_ms': 0.0025}


In [45]:
def build_padded_bitstring(n: int, fill: str = "0") -> str:
    if not isinstance(n, int):
        raise TypeError("n debe ser int")

    if n <= 0:
        raise ValueError("n debe ser > 0")

    if fill not in {"0", "1"}:
        raise ValueError("fill debe ser '0' o '1'")
    return fill *n

bit_string = build_padded_bitstring(N_batch)
bit_string

'00000000'

In [46]:
# Creando un OP RETURN para publicar una REVOCACION

def encode_bitstring_payload(bit_string: str) -> bytes:
    if not isinstance(bit_string, str):
        raise TypeError("bit_string debe ser str")

    if len(bit_string) == 0:
        raise ValueError("bit_string está vacío")

    if any(c not in "01" for c in bit_string):
        raise ValueError("bit_string solo puede contener '0' y '1'")

    bit_len = len(bit_string)

    # Empaquetado a bytes con padding a la derecha solo para almacenamiento
    padded_len = ((bit_len + 7) // 8) * 8
    padded_bits = bit_string.ljust(padded_len, "0")
    packed = int(padded_bits, 2).to_bytes(padded_len // 8, "big")

    # Prefijo de 4 bytes: longitud exacta en bits (big-endian)
    return bit_len.to_bytes(4, "big") + packed


def decode_bitstring_payload(payload: bytes) -> str:
    if not isinstance(payload, bytes):
        raise TypeError("payload debe ser bytes")

    if len(payload) < 4:
        raise ValueError("payload demasiado corto")

    bit_len = int.from_bytes(payload[:4], "big")
    packed = payload[4:]

    if bit_len == 0:
        raise ValueError("bit_len inválido: 0")

    expected_bytes = (bit_len + 7) // 8
    if len(packed) != expected_bytes:
        raise ValueError(
            f"payload inconsistente: se esperaban {expected_bytes} bytes y llegaron {len(packed)}"
        )

    bits = bin(int.from_bytes(packed, "big"))[2:].zfill(len(packed) * 8)
    return bits[:bit_len]


# =========================
# entrada del usuario
# =========================

batch_size = len(leafs)

if len(bit_string) != batch_size:
    raise ValueError(
        f"El bit_string debe tener longitud igual al batch. "
        f"len(bit_string)={len(bit_string)} != batch_size={batch_size}"
    )

# =========================
# UTXO a gastar
# =========================
txid_in = "e6edc2377ed62f78453b5d8daacfdbf825549ea3b489e5bf81edfb64a33fd7f9"
vout_in = 0
amount_in = 1 * 100_000_000

# =========================
# destino normal
# =========================
addr_out = "bcrt1p2mzmka9fgs4pjaj78x08k9cgxrhnts92s7nqnfsr08z6d49jt0fsvcrrpq"
addr_out_obj = P2trAddress.from_address(addr_out)
amount_out = 1 * 500_000

# =========================
# OP_RETURN sin ambigüedad
# payload = [4 bytes bit_len][packed_bits]
# =========================
payload = encode_bitstring_payload(bit_string)
opret_script = Script(["OP_RETURN", payload.hex()])
opret_out = TxOutput(0, opret_script)

# =========================
# fee y cambio
# =========================
fee = 1000
change = amount_in - amount_out - fee

if change < 0:
    raise ValueError("change < 0: fondos insuficientes")

# =========================
# input / outputs
# =========================
txin = TxInput(txid_in, vout_in)

out1 = TxOutput(amount_out, addr_out_obj.to_script_pub_key())
out2 = TxOutput(change, taproot_addr.to_script_pub_key())

tx = Transaction(
    [txin],
    [out1, opret_out, out2],
    has_segwit=True
)

# =========================
# firma key-path Taproot
# =========================
utxo_scripts = [taproot_addr.to_script_pub_key()]
amounts = [amount_in]

sig = btc_sk.sign_taproot_input(
    tx,
    0,
    utxo_scripts,
    amounts,
    script_path=False,
    tapleaf_scripts=tap_tree,
)

tx.witnesses.append(TxWitnessInput([sig]))

rawtx = tx.serialize()
print(rawtx)
print("TXID:", tx.get_txid())


02000000000101f9d73fa364fbed81bfe589b4a39e5425f8dbcfaa8d5d3b45782fd67e37c2ede60000000000fdffffff0320a107000000000022512056c5bb74a9442a19765e399e7b170830ef35c0aa87a609a60379c5a6d4b25bd30000000000000000076a050000000800f83bee05000000002251207e5a70a31b53afab2b52cd35cacad10c4ee5efccb894b1e9feb1f86d667b096c01409daeaef8b9e27bb4706a01402556326c694f86896ab8f68f90c0383e193f2125030df3acac1277586284753c0ec162bf2209b3cc2f6b99cc655f5428d9a79dc200000000
TXID: 246c880f5c0c2a1dd4e1cde562a2a2bb2fc100a849054eef6a94d39ac8105880


In [47]:
from bitcoinrpc.authproxy import AuthServiceProxy
from pprint import pprint


RPC_USER = "ghost"
RPC_PASS = "ghost"
RPC_PORT = 18443
RPC_HOST = "127.0.0.1"

rpc = AuthServiceProxy(f"http://{RPC_USER}:{RPC_PASS}@{RPC_HOST}:{RPC_PORT}")



decoded = rpc.decoderawtransaction(rawtx)

print("vsize =", decoded["vsize"])
print("size =", decoded["size"])

vsize = 170
size = 221


In [48]:
def tx_size_metrics(tx):
    raw = tx.serialize()
    raw_bytes = bytes.fromhex(raw)
    return {
        "hex_len": len(raw),
        "bytes": len(raw_bytes),
    }
    
metrics = tx_size_metrics(tx)
print(metrics)

{'hex_len': 442, 'bytes': 221}


In [49]:
def decode_opreturn_bitstring(opreturn_hex: str) -> str:
    raw = bytes.fromhex(opreturn_hex)

    if len(raw) < 2:
        raise ValueError("script demasiado corto")

    if raw[0] != 0x6A:
        raise ValueError("no es OP_RETURN")

    push_len = raw[1]
    payload = raw[2:2 + push_len]

    if len(payload) != push_len:
        raise ValueError("payload incompleto")

    if len(payload) < 4:
        raise ValueError("payload demasiado corto para contener bit_len")

    bit_len = int.from_bytes(payload[:4], "big")
    packed = payload[4:]

    expected_bytes = (bit_len + 7) // 8
    if len(packed) != expected_bytes:
        raise ValueError("payload inconsistente")

    bits = bin(int.from_bytes(packed, "big"))[2:].zfill(len(packed) * 8)
    return bits[:bit_len]


In [50]:
opreturn = "6a050000000580"
bit_string = decode_opreturn_bitstring(opreturn)
print(bit_string)

10000
